In [ ]:
#| default_exp window

In [ ]:
#| hide
import socket
from http.server import BaseHTTPRequestHandler, HTTPServer
from importlib.util import find_spec
from threading import Event, Thread
from fastcore.test import *
from nbdev.showdoc import *

Open a native window over a loopback web server and integrate platform menus and file-open events.

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import os, sys, threading, time

In [ ]:
#| export
from fastcore.all import Path

In [ ]:
#| export
from fastcore.xdg import xdg_config_home

In [ ]:
#| export
from kavacha.menus import Binding, Js, MenuItem, Std

In [ ]:
#| export
APP_NAME = os.environ.get('KAVACHA_APP_NAME') or 'app'
ENV_PREFIX = 'KAVACHA_'

def use_app(name, cfg=None, env_prefix='KAVACHA_'):
    "Wear `name`: its config directory, its storage and its environment prefix."
    global APP_NAME, CFG, STORAGE, ENV_PREFIX, EAGER_CDP
    APP_NAME, ENV_PREFIX = str(name or 'app'), str(env_prefix or '')
    CFG = Path(cfg) if cfg else Path(os.environ.get(f'{ENV_PREFIX}CFG') or xdg_config_home()/APP_NAME)
    STORAGE, EAGER_CDP = CFG/'webview', f'{ENV_PREFIX}CDP_EAGER'
    return APP_NAME

In [ ]:
#| export
CFG = Path(os.environ.get('KAVACHA_CFG') or xdg_config_home()/APP_NAME)

`APP_NAME` and `CFG` are read from the environment once, at import. A host that wants its own name or its own configuration directory sets `$KAVACHA_APP_NAME` or `$KAVACHA_CFG` before importing this module. Setting either afterwards changes nothing.

Reading them creates nothing on disk.

In [ ]:
#| export
def errstr(e): return f"{type(e).__name__}: {e}"

`errstr` includes the exception type so empty and ambiguous messages remain useful.

In [ ]:
#| exec_doc
errstr(ValueError('port 9223 is in use')), errstr(KeyError('cocoa'))

In [ ]:
#| export
BACKENDS = {'darwin': 'cocoa', 'win32': 'edgechromium', 'linux': 'gtk'}

In [ ]:
#| export
DEFAULT_SIZE = (1440, 900)

In [ ]:
#| export
MIN_SIZE = (900, 560)

In [ ]:
#| export
STORAGE = CFG/'webview'

In [ ]:
#| export
EAGER_CDP = 'KAVACHA_CDP_EAGER'

`DEFAULT_SIZE` sets the initial window size. `MIN_SIZE` limits both requested and user-resized dimensions.

In [ ]:
#| exec_doc
DEFAULT_SIZE, MIN_SIZE, str(STORAGE).replace(str(CFG), '/cfg')

In [ ]:
#| export
class ShellApi:
    "What the page may ask the native window for, as `window.pywebview.api.*`."
    on_open_folder = None
    def open_folder(self, path):
        "Open a folder in a window of its own. pywebview runs every `js_api` call on its own thread."
        if not path or self.on_open_folder is None: return False
        self.on_open_folder(str(path))
        return True
    def pick_folder(self):
        "The platform's own folder chooser. A path, or None if it was cancelled."
        import webview
        if (window := webview.active_window()) is None: return None
        try: picked = window.create_file_dialog(webview.FileDialog.FOLDER)
        except Exception as e:
            print(f'  folder dialog: {errstr(e)}')
            return None
        return str(picked[0]) if picked else None
    on_recent = None
    def recent(self):
        "Whatever the host offers as recent folders, or nothing."
        return list(self.on_recent()) if self.on_recent else []

`ShellApi` is the page's native API. pywebview may call its methods from worker threads.

In [ ]:
#| exec_doc
api = ShellApi()
opened = []
api.on_open_folder = opened.append
api.open_folder('/proj/demo'), api.open_folder(''), opened

In [ ]:
#| hide
test_eq(ShellApi().open_folder('/proj/demo'), False)   
test_eq(ShellApi().recent(), [])
api.on_recent = lambda: iter(['/proj/demo', '/proj/notes'])
test_eq(api.recent(), ['/proj/demo', '/proj/notes'])   
api.open_folder(Path('/proj/notes'))
test_eq(opened[-1], '/proj/notes')                     

In [ ]:
#| export
def backend(platform=None):
    "pywebview's GUI name for `platform`, or None where Leela has no supported webview."
    return BACKENDS.get(platform or sys.platform)

In [ ]:
#| export
def shell_ready(platform=None):
    "`(ok, why)`: whether a native window can be opened here, and what is missing if not."
    gui = backend(platform)
    if gui is None: return False, f'no native webview backend for {platform or sys.platform}'
    try: import webview  # noqa: F401
    except ImportError as e: return False, f'pywebview is not installed ({errstr(e)})'
    return True, gui

`shell_ready` reports platform and pywebview availability and returns the selected GUI backend.

In [ ]:
#| exec_doc
backend('darwin'), backend('win32'), backend('plan9')

In [ ]:
#| exec_doc
test_eq(shell_ready('plan9'), (False, 'no native webview backend for plan9'))
ok, why = shell_ready('darwin')
test_eq(ok, why == 'cocoa')

In [ ]:
#| export
def wait_for_http(url, timeout=60, interval=.1):
    "Wait for `url` until `timeout`; return whether it answered."
    import urllib.request
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(url, timeout=.5).close()
            return True
        except urllib.error.HTTPError: return True
        except Exception: time.sleep(interval)
    return False

`wait_for_http` returns when the server responds or the timeout expires; request errors do not escape.

In [ ]:
#| hide
#| exec_doc
class _Root(BaseHTTPRequestHandler):
    def do_GET(self): self.send_response(200 if self.path == '/' else 404); self.end_headers()
    def log_message(self, *a): pass
srv = HTTPServer(('127.0.0.1', 0), _Root); Thread(target=srv.serve_forever, daemon=True).start()
url = f'http://127.0.0.1:{srv.server_port}/'
s = socket.socket(); s.bind(('127.0.0.1', 0)); dead = f'http://127.0.0.1:{s.getsockname()[1]}/'; s.close()

In [ ]:
#| exec_doc
wait_for_http(url), wait_for_http(url + 'missing', timeout=.3)

In [ ]:
#| hide
test_eq(wait_for_http(dead, timeout=.3, interval=.05), False) 

In [ ]:
#| export
def start_cdp(port=9223, headless=True, wait=False, timeout=None, profile=None):
    "Bring up the persistent CDP Chrome the inline browser renders into, on a thread."
    def boot():
        try:
            from fossick.cdp import _debug_ready, cdp_setup
            from fossick.core import syncy
            if _debug_ready(port): return
            syncy(cdp_setup(port=port, headless=headless, timeout=timeout, user_data_dir=profile))
        except Exception as e: print(f'  inline browser unavailable: {errstr(e)}')
    t = threading.Thread(target=boot, daemon=True, name='kavacha-cdp-boot')
    t.start()
    if wait: t.join(timeout or 30)
    return t

In [ ]:
#| export
def warm_cdp(port=9223, profile=None):
    "Start Chrome at launch only if `$KAVACHA_CDP_EAGER` asks for it. See `docs/desktop.md`."
    if os.environ.get(EAGER_CDP, '').strip() not in ('1', 'true', 'yes'): return None
    return start_cdp(port, headless=True, profile=profile)

`start_cdp` starts optional fossick browser support on a daemon thread. Its failure does not stop the application.

In [ ]:
#| hide
if find_spec('fossick') is None:
    for v in ('1', 'true', 'yes', ' yes '):
        os.environ[EAGER_CDP] = v
        t = warm_cdp(); assert t is not None, v
        t.join(10); assert not t.is_alive(), 'the boot thread ends rather than hanging'
for v in ('', '0', 'no', 'TRUE'):
    os.environ[EAGER_CDP] = v
    test_is(warm_cdp(), None)
os.environ.pop(EAGER_CDP, None)
test_is(warm_cdp(), None)

In [ ]:
#| export
def window_size(spec=None):
    "`WIDTHxHEIGHT` from `spec` or `$<prefix>WINDOW_SIZE`, else the default. See `use_app`."
    spec = spec or os.environ.get(f'{ENV_PREFIX}WINDOW_SIZE', '')
    try:
        w, h = (int(n) for n in str(spec).lower().split('x', 1))
        if w >= MIN_SIZE[0] and h >= MIN_SIZE[1]: return w, h
    except (ValueError, TypeError): pass
    return DEFAULT_SIZE

`window_size` reads `WIDTHxHEIGHT` from the specification or `KAVACHA_WINDOW_SIZE` and enforces `MIN_SIZE`.

In [ ]:
#| exec_doc
window_size('1600x1000'), window_size('1280X800'), window_size('320x200'), window_size('wide')

In [ ]:
#| hide
os.environ['KAVACHA_WINDOW_SIZE'] = '1000x700'
test_eq(window_size(), (1000, 700))
test_eq(window_size('1600x1000'), (1600, 1000)) 
os.environ['KAVACHA_WINDOW_SIZE'] = 'huge'
test_eq(window_size(), DEFAULT_SIZE)
del os.environ['KAVACHA_WINDOW_SIZE']
test_eq(window_size(), DEFAULT_SIZE)
test_eq(window_size('900x560'), MIN_SIZE)       
test_eq(window_size('900x559'), DEFAULT_SIZE)
test_eq(window_size('1600x1000x900'), DEFAULT_SIZE)
test_eq(window_size((1600, 1000)), DEFAULT_SIZE)

In [ ]:
#| export
SPLASH = """<!doctype html><meta charset=utf-8><title>{name}</title>
<style>html,body{{height:100%;margin:0;background:#171b20;color:#f4f7f9;
font:15px/1.5 -apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
display:flex;align-items:center;justify-content:center}}
div{{text-align:center;opacity:.85}}b{{display:block;font-size:34px;letter-spacing:-.02em;margin-bottom:10px}}
i{{font-style:normal;font-size:13px;opacity:.6}}</style>
<div><b>{name}</b><i style="{style}">{note}</i></div>"""

def splash(name=None, note='starting the workspace…', style=''):
    "The holding page a window shows before its URL answers, wearing the host's name."
    return SPLASH.format(name=name or APP_NAME, note=note, style=style)

In [ ]:
#| hide
import kavacha.window as _w
test_eq(_w.APP_NAME, 'app')
assert '<b>app</b>' in _w.splash() and '<title>app</title>' in _w.splash()
try:
    _w.use_app('leela', None, 'LEELA_')
    assert '<b>leela</b>' in _w.splash() and '<title>leela</title>' in _w.splash()
    test_eq(_w.EAGER_CDP, 'LEELA_CDP_EAGER')
    test_eq(_w.STORAGE.parent.name, 'leela')
    assert 'did not answer' in _w._unreachable('http://x')
finally: _w.use_app('app')
test_eq((_w.APP_NAME, _w.EAGER_CDP), ('app', 'KAVACHA_CDP_EAGER'))

In [ ]:
#| export
def _unreachable(url):
    "The same splash, saying why nothing arrived."
    return splash(note=f'the workspace at {url} did not answer', style='color:#e06c75')

The self-contained splash is shown until the local server is ready.

In [ ]:
#| exec_doc
_unreachable('http://127.0.0.1:8000/').splitlines()[-1]

In [ ]:
#| hide
html = _unreachable('http://127.0.0.1:8000/')
assert 'starting the workspace' not in html, 'the waiting line is replaced, not appended'
assert 'http' not in SPLASH, 'the splash fetches nothing'
test_eq(len(html.splitlines()), len(SPLASH.splitlines()))

In [ ]:
#| export
DOCK_RECENT = 8

In [ ]:
#| export
_dock_target = None

In [ ]:
#| export
def _recent_target(on_folder):
    "One for the life of the app: a menu holds no reference to a Python object, so this must."
    global _dock_target
    if _dock_target is None: _dock_target = _RecentTarget.alloc().init_with(on_folder)
    return _dock_target

In [ ]:
#| export
def dock_menu(on_folder, recent=None):
    "The Dock's own menu: the folders last open, newest first. `recent` is the host's list of paths."
    from AppKit import NSMenu, NSMenuItem
    rows = list(recent or ())[:DOCK_RECENT]
    menu = NSMenu.alloc().init()
    if not rows:
        menu.addItem_(_disabled('No recent folders'))
        return menu
    target = _recent_target(on_folder)
    for path in rows:
        path = str(path)
        item = NSMenuItem.alloc().initWithTitle_action_keyEquivalent_(
            os.path.basename(path.rstrip('/')) or path, b'openRecent:', '')
        item.setToolTip_(path)
        item.setRepresentedObject_(path)
        item.setTarget_(target)
        menu.addItem_(item)
    return menu

In [ ]:
#| export
def _disabled(title):
    from AppKit import NSMenuItem
    item = NSMenuItem.alloc().initWithTitle_action_keyEquivalent_(title, None, '')
    item.setEnabled_(False)
    return item

The Dock menu preserves the host's recent-folder order and limits it to `DOCK_RECENT` entries.

In [ ]:
#| export
_hidden, _on_folder, _delegates, _on_recent = [], None, None, None

In [ ]:
#| export
def _dock_delegates():
    "The two delegate subclasses. Built once: an Objective-C class name is global to the runtime."
    global _delegates
    if _delegates is not None: return _delegates
    from Foundation import NO, YES
    from webview.platforms.cocoa import BrowserView
    class Windows(BrowserView.WindowDelegate):
        def windowShouldClose_(self, window):
            window.orderOut_(None)
            if window in _hidden: _hidden.remove(window)
            _hidden.append(window)
            return NO
    class App(BrowserView.AppDelegate):
        def applicationShouldHandleReopen_hasVisibleWindows_(self, app, visible):
            if not visible and _hidden: _hidden.pop().makeKeyAndOrderFront_(None)
            return YES
        def applicationDockMenu_(self, app):
            try: return dock_menu(_on_folder, _on_recent() if _on_recent else None)
            except Exception as e: print(f'  dock menu: {errstr(e)}'); return None
    _delegates = (Windows, App)
    return _delegates

On macOS, closing the last window hides it; Dock activation restores the most recently hidden window. Quit still terminates the app.

In [ ]:
#| export
NO_SUBSTITUTION = ('NSAutomaticQuoteSubstitutionEnabled', 'NSAutomaticDashSubstitutionEnabled',
                   'NSAutomaticTextReplacementEnabled', 'NSAutomaticSpellingCorrectionEnabled',
                   'NSAutomaticPeriodSubstitutionEnabled', 'NSAutomaticCapitalizationEnabled',
                   'NSAutomaticInlinePredictionEnabled')

In [ ]:
#| export
def quiet_text_substitution():
    "Turn off the system's autocorrect for this app. True when it took."
    if sys.platform != 'darwin': return False
    try:
        from Foundation import NSUserDefaults
        d = NSUserDefaults.standardUserDefaults()
        for k in NO_SUBSTITUTION: d.setBool_forKey_(False, k)
        return True
    except Exception as e:
        print(f'  the system may still autocorrect what you type: {errstr(e)}')
        return False

`quiet_text_substitution` disables editor-hostile substitutions in this application's defaults domain only.

In [ ]:
#| export
def keep_running_in_dock(on_folder=None, recent=None):
    "Hide the last window rather than quit, and fill the Dock menu. Reopen shows one window. Cmd+Q quits."
    global _on_folder, _on_recent
    try:
        from webview.platforms.cocoa import BrowserView
        _on_folder, _on_recent = on_folder, recent
        BrowserView.WindowDelegate, BrowserView.AppDelegate = _dock_delegates()
        return True
    except Exception as e:
        print(f'  the last window will quit {APP_NAME}: {errstr(e)}')
        return False

`keep_running_in_dock` installs the macOS delegate behavior before any window is created.

In [ ]:
#| hide
if sys.platform != 'darwin': test_eq(quiet_text_substitution(), False)
if find_spec('webview') is None:
    test_eq(keep_running_in_dock(print), False)
    test_is(_on_folder, None)                  

In [ ]:
#| export
WINDOW_KW = dict(min_size=MIN_SIZE, background_color='#171b20', text_select=True, zoomable=True,
                 easy_drag=False)

`WINDOW_KW` is what every window is built with. The title and the size are the two things a caller may vary and are not in it, so a window opened at launch and one opened for a folder an hour later are the same window.

`min_size` is `MIN_SIZE`, the floor `window_size` enforces.

In [ ]:
#| exec_doc
WINDOW_KW

In [ ]:
#| hide
test_is(WINDOW_KW['min_size'], MIN_SIZE)
assert not {'title', 'width', 'height'} & set(WINDOW_KW), 'the per-window arguments stay out of it'

In [ ]:
#| export
_api = None

In [ ]:
#| export
_windows = {}

In [ ]:
#| export
def shell_api():
    "The one `ShellApi` every window shares."
    global _api
    if _api is None: _api = ShellApi()
    return _api

All windows share one `ShellApi`; `run_shell` installs the host callbacks before opening them.

In [ ]:
#| exec_doc
shell_api() is shell_api()

In [ ]:
#| export
def make_window(title=None, size=None):
    "A window on the splash, waiting for a URL."
    import webview
    w, h = size or window_size()
    STORAGE.mkdir(parents=True, exist_ok=True)
    return webview.create_window(title or APP_NAME, html=splash(title or APP_NAME), width=w, height=h,
                                 js_api=shell_api(), **WINDOW_KW)

`make_window` opens the splash immediately and points the window at its workspace after the server responds.

In [ ]:
#| export
def off_main_thread(fn, name='kavacha-shell'):
    "Run `fn(arg)` on a thread of its own; `create_window` builds nothing on the main thread."
    def go(arg): threading.Thread(target=fn, args=(arg,), daemon=True, name=name).start()
    return go

`off_main_thread` runs native callbacks on daemon threads so they do not create windows inside the GUI loop.

In [ ]:
#| exec_doc
seen, done = [], Event()
go = off_main_thread(lambda p: (seen.append((p, threading.current_thread().name)), done.set()))
go('/proj/demo'); done.wait(5)
seen

In [ ]:
#| export
def open_window(url, title=None, key=None, url_wait=90):
    "A window on `url`, raised rather than opened twice when `key` already has one."
    title = title or APP_NAME
    if key and (window := _windows.get(key)) is not None:
        try: window.show()
        except Exception as e: print(f'  could not raise the window for {title}: {errstr(e)}')
        return window
    window = make_window(title)
    if key: _windows[key] = window
    _point(window, url, url_wait)
    return window

In [ ]:
#| export
def _point(window, url, url_wait=90):
    "Wait for the server, then show the workspace. A window that never answers says so."
    if wait_for_http(url, timeout=url_wait): window.load_url(url)
    else: window.load_html(_unreachable(url))

`open_window` raises an existing workspace window or creates one when absent.

In [ ]:
#| hide
#| exec_doc
class _Win:
    "Stands in for the pywebview window: `open_window` and `_point` use three of its methods."
    def __init__(self, fail=False): self.calls, self.fail = [], fail
    def show(self):
        if self.fail: raise RuntimeError('the window is gone')
        self.calls.append('show')
    def load_url(self, u): self.calls.append(('url', u))
    def load_html(self, h): self.calls.append(('html', h))

In [ ]:
#| exec_doc
_windows['demo'], _windows['gone'] = _Win(), _Win(fail=True)
u = 'http://127.0.0.1:8000/'
open_window(u, 'Demo', key='demo').calls, open_window(u, 'Notes', key='gone').calls

In [ ]:
#| hide
w = _Win(); _point(w, url)
test_eq(w.calls, [('url', url)])
w = _Win(); _point(w, dead, url_wait=.3)
test_eq(w.calls[0][0], 'html')
assert dead in w.calls[0][1], 'the window says which address did not answer'
_windows.clear()

In [ ]:
#| export
def run_shell(urls, titles=(), url_wait=90, gui=None, icon=None, size=None, on_ready=None,
              on_open_folder=None, keys=(), bar=(), lookup=None, on_action=None, on_recent=None):
    "Open one native window per workspace URL and block until they close."
    ok, why = shell_ready()
    if not ok: raise RuntimeError(why)
    import webview
    urls = list(urls)
    if not urls: raise ValueError('the desktop shell needs at least one workspace URL')
    titles = list(titles) + [''] * (len(urls) - len(titles))
    keys = list(keys) + [None] * (len(urls) - len(keys))
    open_folder = off_main_thread(on_open_folder, 'kavacha-open-folder') if on_open_folder else None
    shell_api().on_open_folder = open_folder
    if sys.platform == 'darwin':
        quiet_text_substitution()
        keep_running_in_dock(open_folder, on_recent)
    windows = [make_window(t, size) for t in titles]
    for key, window in zip(keys, windows):
        if key: _windows[key] = window
    if open_folder is not None: watch_open_events(open_folder)
    def load(*_):
        "Once the GUI loop is up: wait for the server, then point each window at it."
        for window, url in zip(windows, urls): _point(window, url, url_wait)
        if on_ready is None: return
        try: on_ready()
        except Exception as e: print(f'  desktop shell: {errstr(e)}')
    built = menus(bar, lookup, on_action) if bar else []
    webview.start(load, gui=gui or why, debug=bool(os.environ.get(f'{ENV_PREFIX}WEBVIEW_DEBUG')),
                  private_mode=False, storage_path=str(STORAGE), icon=icon, menu=built)

`run_shell` opens workspace windows and owns the native GUI loop. Call it on the main thread.

In [ ]:
#| hide
if find_spec('webview') is None:
    test_fail(lambda: run_shell(['http://127.0.0.1:8000/']), contains='pywebview is not installed',
              exc=RuntimeError)

In [ ]:
#| export
def menus(bar, lookup, on_action, run=None):
    "Build the macOS menu bar, or return an empty list."
    if sys.platform != 'darwin': return []
    try:
        import webview
        webview.settings['SHOW_DEFAULT_MENUS'] = False
        built, specs = menu_bar(bar, lookup, on_action, run)
        install_menu_patch(specs, lookup)
        return built
    except Exception as e:
        print(f'  no menu bar: {errstr(e)}')
        return []

`menus` returns the macOS menu bar, or an empty list when unavailable.

In [ ]:
#| hide
if sys.platform != 'darwin': test_eq(menus(), [])

In [ ]:
#| export
def watch_open_events(on_folder):
    "Answer the `odoc` Apple Event Finder sends a running app. macOS only, best-effort."
    if sys.platform != 'darwin': return None
    try: from Foundation import NSAppleEventManager, NSURL
    except ImportError: return None
    def code(s): return int.from_bytes(s.encode(), 'big')
    def handle(event, _reply):
        try:
            items = event.paramDescriptorForKeyword_(code('----'))
            for i in range(1, (items.numberOfItems() or 0) + 1):
                url = items.descriptorAtIndex_(i).stringValue()
                if not url: continue
                p = NSURL.URLWithString_(url).path() if url.startswith('file:') else url
                if p and os.path.isdir(str(p)): on_folder(str(p))
        except Exception as e: print(f'  open-folder event: {errstr(e)}')
    try:
        mgr = NSAppleEventManager.sharedAppleEventManager()
        mgr.setEventHandler_andSelector_forEventClass_andEventID_(
            _OpenHandler.alloc().init_with(handle), b'handleEvent:reply:',
            code('aevt'), code('odoc'))
        return handle
    except Exception as e:
        print(f'  could not register the open-folder handler: {errstr(e)}')
        return None

`watch_open_events` handles macOS folder-open Apple Events and forwards paths off the main thread.

In [ ]:
#| hide
if sys.platform != 'darwin': test_is(watch_open_events(print), None)

In [ ]:
#| export
try:                                     # pragma: no cover - macOS only
    import objc
    from Foundation import NSObject
    class _RecentTarget(NSObject):
        "A target for a Dock menu row."
        def init_with(self, fn):
            self = objc.super(_RecentTarget, self).init()
            if self is None: return None
            self._fn = fn
            return self
        def openRecent_(self, sender):
            path = sender.representedObject()
            if self._fn and path: self._fn(str(path))

    class _OpenHandler(NSObject):
        "An Apple Event handler must be a selector on an Objective-C object."
        def init_with(self, fn):
            self = objc.super(_OpenHandler, self).init()
            if self is None: return None
            self._fn = fn
            return self
        def handleEvent_reply_(self, event, reply): self._fn(event, reply)
except Exception:                        # pragma: no cover - everywhere else
    _OpenHandler = _RecentTarget = None



The Objective-C event and menu targets are `None` when pyobjc is unavailable; callers guard them by platform.

In [ ]:
#| hide
if find_spec('objc') is None: test_is(_RecentTarget, None); test_is(_OpenHandler, None)

In [ ]:
#| export
CMD, CTRL, ALT, SHIFT = 1 << 20, 1 << 18, 1 << 19, 1 << 17

In [ ]:
#| export
_MODS = {'mod': CMD, 'cmd': CMD, 'ctrl': CTRL, 'alt': ALT, 'opt': ALT, 'shift': SHIFT,
         'hyper': CMD | CTRL | ALT | SHIFT}

In [ ]:
#| export
_NAMED = {'up': '', 'down': '', 'left': '', 'right': '',
          'pageup': '', 'pagedown': '', 'home': '', 'end': '',
          'delete': '', 'enter': '\r', 'return': '\r', 'tab': '\t', 'space': ' ',
          'backspace': '\x08', 'escape': '\x1b'}

Modifier constants are defined locally so menu chords can be parsed on any platform.

In [ ]:
#| export
def mac_key(chord):
    "`(keyEquivalent, modifierMask)` for one chord, or None when AppKit cannot spell it."
    parts = str(chord or '').split()[0].split('+') if chord else []
    if not parts or not parts[-1]: return None
    mask, base = 0, parts[-1]
    for p in parts[:-1]:
        if (m := _MODS.get(p.lower())) is None: return None
        mask |= m
    if len(base) == 1 and base.isupper(): mask |= SHIFT
    base = _NAMED.get(base.lower(), base.lower())
    if len(base) != 1: return None
    return base, mask

`mac_key` converts a supported chord into AppKit key and modifier values; unsupported chords return `None`.

In [ ]:
#| exec_doc
mac_key('mod+p'), mac_key('mod+f5'), mac_key('meta+p')

In [ ]:
#| exec_doc
test_eq(mac_key('mod+P'), mac_key('mod+shift+p'))     
test_eq(mac_key('hyper+k'), ('k', CMD | CTRL | ALT | SHIFT))
test_eq(mac_key('mod+k mod+s'), mac_key('mod+k'))     

In [ ]:
#| hide
test_eq(mac_key('mod+tab'), ('\t', CMD))
test_eq(mac_key('escape'), ('\x1b', 0))               
test_eq(len(mac_key('mod+up')[0]), 1)                 
for bad in (None, '', 'mod+', 'mod+f5', 'meta+p', 'mod+ctrl+f12'): test_is(mac_key(bad), None)

In [ ]:
#| export
def menu_chord(action, lookup):
    "Return a global menu chord that may pre-empt the web view."
    k = lookup(action)
    if k is None or not k.bound or k.scope != 'global': return None
    first = k.keys[0]
    if not (set(first.split('+')[:-1]) & {'mod', 'cmd', 'ctrl', 'alt', 'opt', 'hyper'}): return None
    return mac_key(first)

`menu_chord` exposes only global, bound, macOS-compatible chords that should pre-empt the web view.

In [ ]:
#| export
def _title(row, lookup):
    "The label for an action row: the keymap's, or the action with its underscores opened up."
    k = lookup(row.action)
    t = (k.label if k and k.label else row.action.replace('_', ' '))
    return t[:1].upper() + t[1:]

In [ ]:
#| exec_doc
KEYS = {'save':         Binding(('mod+s',), 'Save'),
        'quick_open':   Binding(('mod+shift+o',)),
        'find_in_file': Binding(('mod+f',), 'Find', scope='editor'),
        'comment':      Binding(('/',), 'Comment'),
        'orphan':       Binding()}
{a: menu_chord(a, KEYS.get) for a in ('save', 'quick_open', 'find_in_file', 'comment', 'orphan')}

`save` is global, bound and modified, so it earns its chord. The other three are each refused for a different reason, and an action the keymap has never heard of is refused too.

In [ ]:
#| hide
test_eq(menu_chord('save', KEYS.get), ('s', CMD))
test_eq(menu_chord('quick_open', KEYS.get), ('o', CMD | SHIFT))
test_is(menu_chord('find_in_file', KEYS.get), None)
test_is(menu_chord('comment', KEYS.get), None)     
test_is(menu_chord('orphan', KEYS.get), None)      
test_is(menu_chord('never_heard_of_it', KEYS.get), None)

`_title` prefers the keymap's label. Without one it opens up the action's underscores, so a row is never blank and never shows a bare identifier.

In [ ]:
#| exec_doc
_title(MenuItem('save'), KEYS.get), _title(MenuItem('quick_open'), KEYS.get)

In [ ]:
#| hide
test_eq(_title(MenuItem('save'), KEYS.get), 'Save')       
test_eq(_title(MenuItem('quick_open'), KEYS.get), 'Quick open')
test_eq(_title(MenuItem('never_seen'), KEYS.get), 'Never seen')

In [ ]:
#| export
def menu_bar(bar, lookup, on_action, run=None):
    "pywebview menus for `bar`, plus the table `_decorate` needs to finish them off AppKit-side."
    from webview.menu import Menu, MenuAction, MenuSeparator
    run = run or _menu_run
    menus, specs = [], {}
    for title, rows in bar:
        items = []
        for row in rows:
            if isinstance(row, MenuItem) and not row.action: items.append(MenuSeparator()); continue
            if isinstance(row, Std):
                specs[(title, row.title)] = row
                items.append(MenuAction(row.title, lambda: None))
            elif isinstance(row, Js):
                items.append(MenuAction(row.title, (lambda e: lambda: run(e))(row.expr)))
            else:
                name = _title(row, lookup)
                specs[(title, name)] = row
                items.append(MenuAction(name, (lambda a: lambda: on_action(a))(row.action)))
        menus.append(Menu(title, items))
    return menus, specs

`menu_bar` builds pywebview menus and returns the row lookup used for AppKit decoration.

In [ ]:
#| export
def eval_js(js):
    "Run one expression in the window that has focus, and return what it evaluated to."
    import webview
    if (window := webview.active_window()) is None: return None
    try: return window.evaluate_js(js)
    except Exception as e:
        print(f'  menu: {errstr(e)}')
        return None

_menu_run = eval_js

In [ ]:
#| export
_menu_patched = False

In [ ]:
#| export
def install_menu_patch(specs, lookup):
    "Reapply AppKit decoration whenever pywebview rebuilds the menu."
    global _menu_patched
    if _menu_patched: return False
    try:
        from webview.platforms.cocoa import BrowserView
    except Exception as e:
        print(f'  no menu bar: {errstr(e)}')
        return False
    original = BrowserView._recreate_menus
    def recreate(self, user_menu):
        main = original(self, user_menu)
        try: _decorate(main, specs, lookup)
        except Exception as e: print(f'  menu bar: {errstr(e)}')
        return main
    BrowserView._recreate_menus = recreate
    _menu_patched = True
    return True

`install_menu_patch` reapplies AppKit decoration whenever pywebview rebuilds the menu bar.

In [ ]:
#| hide
if find_spec('webview') is None:
    test_eq(install_menu_patch({}, KEYS.get), False)
    test_eq(_menu_patched, False)

In [ ]:
#| export
def _decorate(main, specs, lookup):
    "Chords onto the rows that may have one, and the responder chain onto the rows AppKit owns."
    import AppKit
    for i in range(main.numberOfItems()):
        sub = main.itemAtIndex_(i).submenu()
        if sub is None: continue
        title = str(sub.title())
        for j in range(sub.numberOfItems()):
            item = sub.itemAtIndex_(j)
            spec = specs.get((title, str(item.title())))
            if spec is None: continue
            if isinstance(spec, Std):
                item.setTarget_(None)
                item.setAction_(spec.selector)
                got = mac_key(f'{spec.mods}+{spec.key}') if spec.key else None
            else: got = menu_chord(spec.action, lookup)
            if got: item.setKeyEquivalent_(got[0]); item.setKeyEquivalentModifierMask_(got[1])
        if title == 'Window': AppKit.NSApp().setWindowsMenu_(sub)

`_decorate` installs responder-chain selectors and allowed key equivalents on native menu items.

In [ ]:
#| hide
srv.shutdown()